<a href="https://colab.research.google.com/github/mobius29er/AIML_Class/blob/main/try_it_20_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Comparing Aggregate Models for Regression

This try-it focuses on utilizing ensemble models in a regression setting.  Much like you have used individual classification estimators to form an ensemble of estimators -- here your goal is to explore ensembles for regression models.  As with your earlier assignment, you will use scikitlearn to carry out the ensembles using the `VotingRegressor`.   


#### Dataset and Task

Below, a dataset containing census information on individuals and their hourly wage is loaded using the `fetch_openml` function.  OpenML is another repository for datasets [here](https://www.openml.org/).  Your task is to use ensemble methods to explore predicting the `wage` column of the data.  Your ensemble should at the very least consider the following models:

- `LinearRegression` -- perhaps you even want the `TransformedTargetRegressor` here.
- `KNeighborsRegressor`
- `DecisionTreeRegressor`
- `Ridge`
- `SVR`

Tune the `VotingRegressor` to try to optimize the prediction performance and determine if the wisdom of the crowd performed better in this setting than any of the individual models themselves.  Report back on your findings and discuss the interpretability of your findings.  Is there a way to determine what features mattered in predicting wages?

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import VotingRegressor
from sklearn.pipeline import Pipeline
from sklearn.datasets import fetch_openml

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import VotingRegressor
from sklearn.pipeline import Pipeline
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error
from sklearn.inspection import permutation_importance

# Load dataset
survey = fetch_openml(data_id=534, as_frame=True).frame

In [3]:
# Display the first few rows to understand the data
survey.head()

,EDUCATION,SOUTH,SEX,EXPERIENCE,UNION,WAGE,AGE,RACE,OCCUPATION,SECTOR,MARR
0,8,no,female,21,not_member,5.10,35,Hispanic,Other,Manufacturing,Married
1,9,no,female,42,not_member,4.95,57,White,Other,Manufacturing,Married
2,12,no,male,1,not_member,6.67,19,White,Other,Manufacturing,Unmarried
3,12,no,male,4,not_member,4.00,22,White,Other,Other,Unmarried
4,12,no,male,17,not_member,7.50,35,White,Other,Other,Married


# Task
Clean the data by identifying and one-hot encoding categorical features, handling missing values, and splitting the data into training and testing sets.

## Identify categorical features

### Subtask:
Identify columns with categorical data that need encoding.


**Reasoning**:
Identify columns with categorical data by iterating through the DataFrame and checking data types.



In [4]:
for column in survey.columns:
    if survey[column].dtype in ['object', 'category']:
        print(column)

SOUTH
SEX
UNION
RACE
OCCUPATION
SECTOR
MARR


## One-hot encode categorical features

### Subtask:
Convert categorical features into numerical format using one-hot encoding.


**Reasoning**:
Convert the identified categorical features into numerical format using one-hot encoding.



In [5]:
categorical_cols = ['SOUTH', 'SEX', 'UNION', 'RACE', 'OCCUPATION', 'SECTOR', 'MARR']
survey_encoded = pd.get_dummies(survey, columns=categorical_cols, drop_first=True)
display(survey_encoded.head())

,EDUCATION,EXPERIENCE,WAGE,AGE,SOUTH_yes,SEX_male,UNION_not_member,RACE_Other,RACE_White,OCCUPATION_Management,OCCUPATION_Other,OCCUPATION_Professional,OCCUPATION_Sales,OCCUPATION_Service,SECTOR_Manufacturing,SECTOR_Other,MARR_Unmarried
0,8,21,5.10,35,False,False,True,False,False,False,True,False,False,False,True,False,False
1,9,42,4.95,57,False,False,True,False,True,False,True,False,False,False,True,False,False
2,12,1,6.67,19,False,True,True,False,True,False,True,False,False,False,True,False,True
3,12,4,4.00,22,False,True,True,False,True,False,True,False,False,False,False,True,True
4,12,17,7.50,35,False,True,True,False,True,False,True,False,False,False,False,True,False


## Handle missing values

### Subtask:
Check for and handle any missing values in the dataset.


**Reasoning**:
Check for missing values in the DataFrame.



In [6]:
missing_values = survey_encoded.isnull().sum()
print(missing_values)

EDUCATION                  0
EXPERIENCE                 0
WAGE                       0
AGE                        0
SOUTH_yes                  0
SEX_male                   0
UNION_not_member           0
RACE_Other                 0
RACE_White                 0
OCCUPATION_Management      0
OCCUPATION_Other           0
OCCUPATION_Professional    0
OCCUPATION_Sales           0
OCCUPATION_Service         0
SECTOR_Manufacturing       0
SECTOR_Other               0
MARR_Unmarried             0
dtype: int64


## Split data

### Subtask:
Split the data into training and testing sets.


**Reasoning**:
Separate features and target variable and split the data into training and testing sets.



In [7]:
X = survey_encoded.drop('WAGE', axis=1)
y = survey_encoded['WAGE']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training set shape:", X_train.shape, y_train.shape)
print("Testing set shape:", X_test.shape, y_test.shape)

Training set shape: (427, 16) (427,)
Testing set shape: (107, 16) (107,)


## Summary:

### Data Analysis Key Findings

*   The categorical features identified for encoding were `SOUTH`, `SEX`, `UNION`, `RACE`, `OCCUPATION`, `SECTOR`, and `MARR`.
*   The categorical features were successfully one-hot encoded, replacing the original columns with new binary columns.
*   No missing values were found in the encoded dataset.
*   The data was successfully split into training and testing sets with a ratio of 80% for training (427 samples) and 20% for testing (107 samples).

### Insights or Next Steps

*   The cleaned and prepared dataset is now ready for model training and evaluation.
*   The next step should involve selecting a suitable regression model to predict the 'WAGE' based on the prepared features and then training and evaluating the chosen model using the split data.


# Task
Analyze the provided data to predict the `wage` column using ensemble methods, specifically a `VotingRegressor`. The ensemble should include `LinearRegression` (potentially with `TransformedTargetRegressor`), `KNeighborsRegressor`, `DecisionTreeRegressor`, `Ridge`, and `SVR`. Tune the `VotingRegressor` to optimize prediction performance and compare its performance against the individual models to determine if the ensemble approach is superior.

## Define individual models

### Subtask:
Define the individual regression models to be used in the ensemble.


**Reasoning**:
Instantiate the required individual regression models.



In [8]:
linear_regression = LinearRegression()
k_neighbors_regressor = KNeighborsRegressor()
decision_tree_regressor = DecisionTreeRegressor(random_state=42)
ridge = Ridge(random_state=42)
svr = SVR()

## Create and tune votingregressor

### Subtask:
Create a `VotingRegressor` with the defined individual models and tune it using `GridSearchCV`.


**Reasoning**:
Create a list of estimators, instantiate and tune the VotingRegressor using GridSearchCV.



In [9]:
estimators = [
    ('lr', linear_regression),
    ('knn', k_neighbors_regressor),
    ('dt', decision_tree_regressor),
    ('ridge', ridge),
    ('svr', svr)
]

voting_regressor = VotingRegressor(estimators=estimators)

param_grid = {
    'weights': [[1, 1, 1, 1, 1], [2, 1, 1, 1, 1], [1, 2, 1, 1, 1], [1, 1, 2, 1, 1], [1, 1, 1, 2, 1], [1, 1, 1, 1, 2]]
}

grid_search = GridSearchCV(voting_regressor, param_grid, cv=5, scoring='neg_mean_squared_error')

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score (negative MSE):", grid_search.best_score_)

Best parameters: {'weights': [1, 1, 1, 2, 1]}
Best cross-validation score (negative MSE): -20.05305352313937


## Train and evaluate individual models

### Subtask:
Train and evaluate each individual model on the training and testing data.


**Reasoning**:
Train and evaluate each individual model using the training and testing data and print their MSE scores.



In [10]:
individual_models = [
    ('Linear Regression', linear_regression),
    ('KNeighbors Regressor', k_neighbors_regressor),
    ('Decision Tree Regressor', decision_tree_regressor),
    ('Ridge', ridge),
    ('SVR', svr)
]

for name, model in individual_models:
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    print(f"{name} Test MSE: {mse:.4f}")

Linear Regression Test MSE: 19.5026
KNeighbors Regressor Test MSE: 26.0062
Decision Tree Regressor Test MSE: 59.5471
Ridge Test MSE: 19.4975
SVR Test MSE: 27.5280


## Train and evaluate votingregressor

### Subtask:
Train and evaluate the tuned `VotingRegressor` on the training and testing data.


**Reasoning**:
Get the best estimator from the grid search, predict on the test set, calculate the MSE, and print the result.



In [11]:
best_voting_regressor = grid_search.best_estimator_
y_pred_voting = best_voting_regressor.predict(X_test)
mse_voting = mean_squared_error(y_test, y_pred_voting)
print(f"Tuned Voting Regressor Test MSE: {mse_voting:.4f}")

Tuned Voting Regressor Test MSE: 21.9568


## Compare model performance

### Subtask:
Compare the performance of the individual models and the `VotingRegressor` using a suitable metric like Mean Squared Error.


**Reasoning**:
Compare the MSE values of the individual models and the Voting Regressor to determine which model performed best on the test set.



In [12]:
print("Individual Model Test MSEs:")
print(f"Linear Regression: {19.5026:.4f}")
print(f"KNeighbors Regressor: {26.0062:.4f}")
print(f"Decision Tree Regressor: {59.5471:.4f}")
print(f"Ridge: {19.4975:.4f}")
print(f"SVR: {27.5280:.4f}")
print("\nTuned Voting Regressor Test MSE:")
print(f"Voting Regressor: {21.9568:.4f}")

print("\nComparison:")
if mse_voting < min([19.5026, 26.0062, 59.5471, 19.4975, 27.5280]):
    print("The Tuned Voting Regressor performed best.")
else:
    print("An individual model performed best or equally well.")


Individual Model Test MSEs:
Linear Regression: 19.5026
KNeighbors Regressor: 26.0062
Decision Tree Regressor: 59.5471
Ridge: 19.4975
SVR: 27.5280

Tuned Voting Regressor Test MSE:
Voting Regressor: 21.9568

Comparison:
An individual model performed best or equally well.


## Analyze feature importance

### Subtask:
Explore ways to determine feature importance for the ensemble model (if possible) or individual models.


**Reasoning**:
Calculate and display the sorted coefficients for the Ridge and Linear Regression models to understand feature importance.



In [13]:
# Feature importance for Ridge model
ridge_coef = pd.Series(ridge.coef_, index=X_train.columns)
sorted_ridge_coef = ridge_coef.abs().sort_values(ascending=False)
print("Ridge Model - Top Feature Importances (Absolute Coefficients):")
display(ridge_coef.loc[sorted_ridge_coef.index])

# Feature importance for Linear Regression model
lr_coef = pd.Series(linear_regression.coef_, index=X_train.columns)
sorted_lr_coef = lr_coef.abs().sort_values(ascending=False)
print("\nLinear Regression Model - Top Feature Importances (Absolute Coefficients):")
display(lr_coef.loc[sorted_lr_coef.index])

Ridge Model - Top Feature Importances (Absolute Coefficients):


,0
OCCUPATION_Management,2.969867
UNION_not_member,-2.125914
SEX_male,1.754821
OCCUPATION_Professional,1.646448
RACE_White,0.789732
OCCUPATION_Service,-0.776785
SECTOR_Manufacturing,0.762019
EDUCATION,0.761107
OCCUPATION_Sales,-0.516526
SOUTH_yes,-0.425641



Linear Regression Model - Top Feature Importances (Absolute Coefficients):


,0
OCCUPATION_Management,3.082299
UNION_not_member,-2.168736
SEX_male,1.763136
OCCUPATION_Professional,1.710179
RACE_White,0.838962
EDUCATION,0.830298
SECTOR_Manufacturing,0.803927
OCCUPATION_Service,-0.766262
OCCUPATION_Sales,-0.497844
SOUTH_yes,-0.423335


## Report findings

### Subtask:
Summarize the findings, discussing the performance comparison and interpretability.


**Reasoning**:
Summarize the findings, discussing the performance comparison and interpretability based on the results from the previous steps.



In [14]:
print("### Summary of Findings")

print("\n#### Performance Comparison")
print("The performance of the individual models and the tuned Voting Regressor was evaluated using Mean Squared Error (MSE) on the test set:")
print(f"- Linear Regression Test MSE: {19.5026:.4f}")
print(f"- KNeighbors Regressor Test MSE: {26.0062:.4f}")
print(f"- Decision Tree Regressor Test MSE: {59.5471:.4f}")
print(f"- Ridge Test MSE: {19.4975:.4f}")
print(f"- SVR Test MSE: {27.5280:.4f}")
print(f"- Tuned Voting Regressor Test MSE: {21.9568:.4f}")

print("\nBased on the MSE, the Ridge regression model achieved the lowest error on the test set, with an MSE of 19.4975. Linear Regression performed very similarly. The tuned Voting Regressor, while better than KNeighbors Regressor, Decision Tree Regressor, and SVR, did not outperform the best individual models (Ridge and Linear Regression) in this instance.")
print("This suggests that in this specific case, the 'wisdom of the crowd' as implemented by the Voting Regressor with the tested weights did not lead to improved performance compared to the best individual linear models.")

print("\n#### Feature Importance and Interpretability")
print("Analyzing the coefficients from the Linear Regression and Ridge models provides insights into the features most influential in predicting wage:")
print("\nRidge Model - Top Feature Importances (Absolute Coefficients):")
display(ridge_coef.loc[sorted_ridge_coef.index])
print("\nLinear Regression Model - Top Feature Importances (Absolute Coefficients):")
display(lr_coef.loc[sorted_lr_coef.index])

print("\nBoth models consistently highlight similar features as having the largest impact on wage prediction, based on the absolute magnitude of their coefficients.")
print("- **OCCUPATION_Management**: This feature has the largest positive coefficient in both models, indicating that individuals in management occupations tend to have significantly higher wages, holding other factors constant.")
print("- **UNION_not_member**: This feature has a large negative coefficient, suggesting that not being a union member is associated with lower wages compared to being a union member. The coefficient's magnitude indicates a substantial difference in wage.")
print("- **SEX_male**: This feature also has a notable positive coefficient, suggesting that being male is associated with higher wages compared to being female, when other factors are equal. This points to a potential gender wage gap in the dataset.")
print("- **OCCUPATION_Professional**: Individuals in professional occupations also appear to earn higher wages based on the positive coefficient, though generally less than those in management.")
print("- **RACE_White**: Being of White race is associated with higher wages compared to the reference group (Hispanic in the one-hot encoding with 'Other' also present), as indicated by the positive coefficient.")
print("- **EDUCATION**: A positive coefficient for education suggests that higher levels of education are associated with higher wages, which is a common finding in wage studies.")

print("\nLinear models like Linear Regression and Ridge offer a high degree of interpretability. The coefficients directly indicate the estimated change in wage for a one-unit increase in a feature, assuming all other features are held constant. The magnitude of the coefficients provides a clear sense of the strength of each feature's relationship with the target variable.")
print("In contrast, models like Decision Tree Regressor are less directly interpretable at a global level (though individual decision paths can be followed). The Voting Regressor, as an ensemble, combines the predictions of multiple models, making it even less straightforward to attribute the final prediction to the influence of specific features in a simple, linear fashion. While permutation importance or other model-agnostic methods could be used to assess feature importance for the ensemble, directly interpreting combined coefficients is not feasible.")

print("\nIn summary, while linear models (Ridge and Linear Regression) provided the best performance on this dataset and offered clear insights into the most influential features for wage prediction, the tested Voting Regressor did not yield superior results. The analysis of coefficients from the linear models revealed that occupation (specifically management and professional roles), union membership status, sex, race, and education level are key factors associated with wage differences in this dataset.")

### Summary of Findings

#### Performance Comparison
The performance of the individual models and the tuned Voting Regressor was evaluated using Mean Squared Error (MSE) on the test set:
- Linear Regression Test MSE: 19.5026
- KNeighbors Regressor Test MSE: 26.0062
- Decision Tree Regressor Test MSE: 59.5471
- Ridge Test MSE: 19.4975
- SVR Test MSE: 27.5280
- Tuned Voting Regressor Test MSE: 21.9568

Based on the MSE, the Ridge regression model achieved the lowest error on the test set, with an MSE of 19.4975. Linear Regression performed very similarly. The tuned Voting Regressor, while better than KNeighbors Regressor, Decision Tree Regressor, and SVR, did not outperform the best individual models (Ridge and Linear Regression) in this instance.
This suggests that in this specific case, the 'wisdom of the crowd' as implemented by the Voting Regressor with the tested weights did not lead to improved performance compared to the best individual linear models.

#### Feature Importance and 

,0
OCCUPATION_Management,2.969867
UNION_not_member,-2.125914
SEX_male,1.754821
OCCUPATION_Professional,1.646448
RACE_White,0.789732
OCCUPATION_Service,-0.776785
SECTOR_Manufacturing,0.762019
EDUCATION,0.761107
OCCUPATION_Sales,-0.516526
SOUTH_yes,-0.425641



Linear Regression Model - Top Feature Importances (Absolute Coefficients):


,0
OCCUPATION_Management,3.082299
UNION_not_member,-2.168736
SEX_male,1.763136
OCCUPATION_Professional,1.710179
RACE_White,0.838962
EDUCATION,0.830298
SECTOR_Manufacturing,0.803927
OCCUPATION_Service,-0.766262
OCCUPATION_Sales,-0.497844
SOUTH_yes,-0.423335



Both models consistently highlight similar features as having the largest impact on wage prediction, based on the absolute magnitude of their coefficients.
- **OCCUPATION_Management**: This feature has the largest positive coefficient in both models, indicating that individuals in management occupations tend to have significantly higher wages, holding other factors constant.
- **UNION_not_member**: This feature has a large negative coefficient, suggesting that not being a union member is associated with lower wages compared to being a union member. The coefficient's magnitude indicates a substantial difference in wage.
- **SEX_male**: This feature also has a notable positive coefficient, suggesting that being male is associated with higher wages compared to being female, when other factors are equal. This points to a potential gender wage gap in the dataset.
- **OCCUPATION_Professional**: Individuals in professional occupations also appear to earn higher wages based on the positive co

## Summary:

### Data Analysis Key Findings

*   The individual models tested achieved the following Mean Squared Error (MSE) on the test set: Linear Regression (19.5026), KNeighbors Regressor (26.0062), Decision Tree Regressor (59.5471), Ridge (19.4975), and SVR (27.5280).
*   The tuned `VotingRegressor` obtained a test MSE of 21.9568.
*   The Ridge regression model achieved the lowest test MSE (19.4975) among all models, slightly outperforming Linear Regression and the tuned `VotingRegressor`.
*   Feature importance analysis using coefficients from the Linear Regression and Ridge models identified 'OCCUPATION_Management', 'UNION_not_member', and 'SEX_male' as the most influential features for predicting wage, based on the absolute magnitude of their coefficients.
*   'OCCUPATION_Management' and 'SEX_male' showed positive associations with wage, while 'UNION_not_member' showed a negative association. Other influential features included 'OCCUPATION_Professional', 'RACE_White', and 'EDUCATION'.

### Insights or Next Steps

*   While ensemble methods like `VotingRegressor` often improve performance, in this case, a simple regularized linear model (Ridge) performed best. Further investigation could explore different ensemble configurations or more advanced boosting/bagging techniques to see if they offer improvements.
*   The identified key features (occupation, union status, sex, race, education) align with common factors influencing wages. Further analysis could explore interactions between these features or investigate potential biases (e.g., gender or racial wage gaps) suggested by the coefficients.
